# Tratamento — cadastro_clientes_kaggle e contratos_apolices_kaggle

Regras aplicadas em todas as colunas:
- **Nenhuma linha é removida** de nenhuma base.
- Valores inválidos (`#N/D`, `-`, `?`, `NULL`, `NA`, vazio, etc.) viram `NaN`.
- Valores fora de um range coerente (outliers/sentinelas como `999999`, `-30`, `1`) viram `NaN`.
- Colunas monetárias: remove `R$`, normaliza milhar/decimal e usa `.` para separar centavos.
- Colunas sim/não: `1` para sim, `0` para não, `NaN` para outlier/inválido.
- `idade`/`data_nascimento`: data formatada em `dd/mm/yyyy`; quando `idade` é inválida/outlier, recalcula pela data de nascimento; quando nem idade nem data são aproveitáveis, fica `NaN`.
- `genero` e `estado_civil` seguem a convenção já usada no projeto (genero: M→1, F→0; estado_civil: solteiro→1, casado→0).
- Colunas categóricas com mais de 2 valores (`tipo_cobertura`, `canal_aquisicao`, `metodo_pagamento`, `escolaridade`) são one-hot encoded.


In [1]:
import re
from datetime import date

import numpy as np
import pandas as pd
from dateutil import parser as dateutil_parser

pd.set_option("display.max_columns", None)


## Funções auxiliares (reaproveitadas do `trat_caue.ipynb`)

In [2]:
INVALID_VALUES = {"?", "null", "n/a", "na", "none", "", "-", "--", "#n/d", "n/d"}


def is_invalid(raw) -> bool:
    """True se o valor for NaN ou um dos tokens de 'sem informação' (#N/D, -, ?, etc.)."""
    if pd.isna(raw):
        return True
    return str(raw).strip().lower() in INVALID_VALUES


_MESES = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"


def parse_date(raw):
    """Normaliza qualquer formato de data para dd/mm/yyyy. Retorna NaN se ausente/incoerente."""
    if is_invalid(raw):
        return np.nan
    text = str(raw).strip()

    if re.fullmatch(rf"\d{{1,2}} {_MESES} \d{{4}}", text, re.IGNORECASE):
        try:
            return dateutil_parser.parse(text, dayfirst=True).strftime("%d/%m/%Y")
        except Exception:
            return np.nan

    parts = re.split(r"[-/.]", text)
    if len(parts) != 3:
        return np.nan

    try:
        a, b, c = parts
        if len(a) == 4:
            year, p2, p3 = int(a), int(b), int(c)
            month, day = (p2, p3) if 1 <= p2 <= 12 else (p3, p2)
        elif len(c) == 4:
            year, p1, p2 = int(c), int(a), int(b)
            if p1 > 12:
                day, month = p1, p2
            elif p2 > 12:
                month, day = p1, p2
            else:
                day, month = p1, p2  # ambíguo: assume dd/mm (padrão brasileiro)
        else:
            return np.nan
        return date(year, month, day).strftime("%d/%m/%Y")
    except (ValueError, TypeError):
        return np.nan


def parse_numero(raw, vmin=None, vmax=None):
    """Converte para float. Inválido ou fora de [vmin, vmax] vira NaN."""
    if is_invalid(raw):
        return np.nan
    try:
        val = float(str(raw).strip())
    except ValueError:
        return np.nan
    if vmin is not None and val < vmin:
        return np.nan
    if vmax is not None and val > vmax:
        return np.nan
    return val


def parse_money(raw, vmin, vmax):
    """
    Converte valores monetários (com/sem 'R$', formato BR '1.234,56' ou US '1234.56') para float.
    Quando não há vírgula, usa vmin para decidir se o ponto é separador de milhar sem centavos
    (ex.: 'R$ 252.000' -> 252000.0, e não 252.0).
    Fora de [vmin, vmax] ou inválido vira NaN.
    """
    if is_invalid(raw):
        return np.nan
    text = str(raw).strip().replace("R$", "").replace(" ", "")
    if text.lower() in INVALID_VALUES:
        return np.nan

    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    elif "." in text:
        try:
            if float(text) < vmin:
                text = text.replace(".", "")
        except ValueError:
            pass

    try:
        val = float(text)
    except ValueError:
        return np.nan
    return val if vmin <= val <= vmax else np.nan


def parse_sim_nao(raw, sim_set, nao_set):
    """Normaliza colunas sim/não para 1/0. Outlier/inválido vira NaN."""
    if is_invalid(raw):
        return np.nan
    text = str(raw).strip().lower()
    if text in sim_set:
        return 1
    if text in nao_set:
        return 0
    return np.nan


def calcular_idade(data_str):
    """Calcula a idade a partir de uma data já normalizada em dd/mm/yyyy."""
    try:
        nascimento = dateutil_parser.parse(data_str, dayfirst=True).date()
        hoje = date.today()
        idade = hoje.year - nascimento.year - ((hoje.month, hoje.day) < (nascimento.month, nascimento.day))
        return float(idade)
    except Exception:
        return np.nan


def calcular_dias(data_str):
    """Calcula quantos dias se passaram desde uma data (dd/mm/yyyy) até hoje."""
    try:
        dt = dateutil_parser.parse(data_str, dayfirst=True).date()
        return float((date.today() - dt).days)
    except Exception:
        return np.nan


def one_hot_encode(df, col, categoria_map):
    """
    Normaliza `col` via `categoria_map` (valor limpo em minúsculas -> categoria canônica)
    e faz one-hot encoding, mantendo uma coluna `<col>_NaN` para valores ausentes/inválidos.
    """
    def normalizar(raw):
        if is_invalid(raw):
            return np.nan
        return categoria_map.get(str(raw).strip().lower(), np.nan)

    normalizado = df[col].apply(normalizar)
    dummies = pd.get_dummies(normalizado, prefix=col, dummy_na=True)
    dummies = dummies.rename(columns={f"{col}_nan": f"{col}_NaN"}).astype(int)
    return pd.concat([df.drop(columns=[col]), dummies], axis=1)


## 1. `cadastro_clientes_kaggle.csv`

In [3]:
df_cad = pd.read_csv("../bases/bases_kaggle/cadastro_clientes_kaggle.csv", dtype=str)
print(df_cad.shape)
df_cad.head()


(99000, 12)


,id_cliente,idade,data_nascimento,genero,estado_civil,tem_filhos,qtd_dependentes,escolaridade,renda_anual,possui_imovel,valor_imovel,tempo_residencia_anos
0,221301632992,48.0,13/06/1978,NaN,Casado,,1,Superior,249999997.50,1,"R$ 474.000,00",20
1,221300567667,48.0,06-13-1978,Masc,Casado,true,2,Medio,"R$86.300,00",1,207000.00,12
2,221301745555,36.0,1990/6/10,F,Married,true,4,Medio,"R$76.300,00",1,"R$ 414.000,00",8
3,221300755350,26.0,2000-06-07,F,solteiro,Sim,4,Fundamental,"R$ 238.250,00",1,"R$174.000,00",8
4,221300980984,58.0,15/06/1968,Feminino,Solteiro,S,2,Superior,"113.900,00",1,"350.000,00",NaN


### `data_nascimento` e `idade`

`data_nascimento` é normalizada para `dd/mm/yyyy`. Sempre que `idade` estiver ausente ou fora
do range coerente (17–96 anos, mesmo critério já usado no projeto), ela é recalculada a partir
da data de nascimento. Quando nenhuma das duas colunas tem informação aproveitável, o valor final
é `NaN` — nenhuma linha é removida.

In [4]:
IDADE_MIN, IDADE_MAX = 17, 96

df_cad["data_nascimento"] = df_cad["data_nascimento"].apply(parse_date)
df_cad["idade"] = df_cad["idade"].apply(lambda x: parse_numero(x, IDADE_MIN, IDADE_MAX))

mask_recalcular = df_cad["idade"].isna() & df_cad["data_nascimento"].notna()
df_cad.loc[mask_recalcular, "idade"] = df_cad.loc[mask_recalcular, "data_nascimento"].apply(calcular_idade)

print(f"Idades recalculadas via data_nascimento: {mask_recalcular.sum()}")
print(f"Idades que permaneceram NaN (sem idade nem data válidas): {df_cad['idade'].isna().sum()}")
df_cad[["idade", "data_nascimento"]].describe(include="all")


Idades recalculadas via data_nascimento: 4958
Idades que permaneceram NaN (sem idade nem data válidas): 287


,idade,data_nascimento
count,98713.000000,93953
unique,NaN,102
top,NaN,13/06/1979
freq,NaN,2878
mean,43.699705,NaN
std,12.301176,NaN
min,18.000000,NaN
25%,35.000000,NaN
50%,44.000000,NaN
75%,52.000000,NaN


In [5]:
# genero: M -> 1, F -> 0 | estado_civil: solteiro -> 1, casado -> 0
MASCULINO = {"m", "masc", "masculino"}
FEMININO = {"f", "fem", "feminino"}
df_cad["genero"] = df_cad["genero"].apply(lambda x: parse_sim_nao(x, MASCULINO, FEMININO))

SOLTEIRO = {"s", "solt", "solteiro", "solteiro(a)", "single"}
CASADO = {"c", "casado", "casado(a)", "married"}
df_cad["estado_civil"] = df_cad["estado_civil"].apply(lambda x: parse_sim_nao(x, SOLTEIRO, CASADO))

df_cad[["genero", "estado_civil"]].apply(lambda s: s.value_counts(dropna=False))


,genero,estado_civil
0.0,46135,53705
1.0,47988,40336
NaN,4877,4959


In [6]:
# tem_filhos: sim -> 1, nao -> 0
SIM = {"1", "sim", "s", "true"}
NAO = {"0", "nao", "não", "n", "false"}
df_cad["tem_filhos"] = df_cad["tem_filhos"].apply(lambda x: parse_sim_nao(x, SIM, NAO))

# qtd_dependentes: numérico, faixa observada 0-4, margem de segurança até 10
df_cad["qtd_dependentes"] = df_cad["qtd_dependentes"].apply(lambda x: parse_numero(x, 0, 10))

# escolaridade: categórica -> one-hot
ESCOLARIDADE_MAP = {"medio": "medio", "fundamental": "fundamental", "superior": "superior", "pos": "pos"}
df_cad = one_hot_encode(df_cad, "escolaridade", ESCOLARIDADE_MAP)

# renda_anual: monetária, faixa coerente 10.000-1.000.000 (sentinelas observados: 99999999, 249999997.5)
df_cad["renda_anual"] = df_cad["renda_anual"].apply(lambda x: parse_money(x, 10_000, 1_000_000))

# possui_imovel: sim/nao (já vem majoritariamente como 1/0)
df_cad["possui_imovel"] = df_cad["possui_imovel"].apply(lambda x: parse_sim_nao(x, {"1"}, {"0"}))

# valor_imovel: monetária, faixa coerente 30.000-1.500.000 (sentinelas observados: 999999999, 2999999997)
df_cad["valor_imovel"] = df_cad["valor_imovel"].apply(lambda x: parse_money(x, 30_000, 1_500_000))

# tempo_residencia_anos: numérico, faixa observada 0-40, margem de segurança até 60
df_cad["tempo_residencia_anos"] = df_cad["tempo_residencia_anos"].apply(lambda x: parse_numero(x, 0, 60))

df_cad.describe(include="all")


,id_cliente,idade,data_nascimento,genero,estado_civil,tem_filhos,qtd_dependentes,renda_anual,possui_imovel,valor_imovel,tempo_residencia_anos,escolaridade_fundamental,escolaridade_medio,escolaridade_pos,escolaridade_superior,escolaridade_NaN
count,99000,98713.000000,93953,94123.000000,94041.000000,94033.000000,94049.000000,93771.000000,94094.000000,9.344700e+04,94095.000000,99000.000000,99000.000000,99000.00000,99000.000000,99000.000000
unique,99000,NaN,102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,221301632992,NaN,13/06/1979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,2878,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,43.699705,NaN,0.509844,0.428919,0.526358,1.313826,80270.189611,0.992518,2.985812e+05,6.986790,0.114424,0.377535,0.11503,0.343162,0.049848
std,NaN,12.301176,NaN,0.499906,0.494924,0.499307,1.488352,42697.328965,0.086174,1.228034e+05,6.871146,0.318327,0.484773,0.31906,0.474767,0.217633
min,NaN,18.000000,NaN,0.000000,0.000000,0.000000,0.000000,10200.000000,0.000000,3.000000e+04,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
25%,NaN,35.000000,NaN,0.000000,0.000000,0.000000,0.000000,52000.000000,1.000000,2.240000e+05,2.000000,0.000000,0.000000,0.00000,0.000000,0.000000
50%,NaN,44.000000,NaN,1.000000,0.000000,1.000000,1.000000,71100.000000,1.000000,2.910000e+05,5.000000,0.000000,0.000000,0.00000,0.000000,0.000000
75%,NaN,52.000000,NaN,1.000000,1.000000,1.000000,3.000000,97600.000000,1.000000,3.600000e+05,10.000000,0.000000,1.000000,0.00000,1.000000,0.000000


In [7]:
print("Linhas antes:", 99000, "| Linhas depois:", len(df_cad))
assert len(df_cad) == 99000, "Nenhuma linha deveria ter sido removida"

df_cad.to_csv("../bases/bases_kaggle/tratadas/cadastro_clientes_kaggle_tratada.csv", index=False)
print("Salvo em ../bases/bases_kaggle/tratadas/cadastro_clientes_kaggle_tratada.csv")


Linhas antes: 99000 | Linhas depois: 99000


Salvo em ../bases/bases_kaggle/tratadas/cadastro_clientes_kaggle_tratada.csv


## 2. `contratos_apolices_kaggle.csv`

Arquivo separado por `;`.

In [8]:
df_cont = pd.read_csv("../bases/bases_kaggle/contratos_apolices_kaggle.csv", sep=";", dtype=str)
print(df_cont.shape)
df_cont.head()


(94000, 13)


,cod_individuo,num_apolices_ativas,tipo_cobertura,valor_premio_anual,tempo_cliente_dias,data_primeira_apolice,num_produtos_contratados,valor_cobertura_total,franquia_media,canal_aquisicao,pagamento_em_dia,desconto_aplicado_pct,metodo_pagamento
0,IND-221302198005,1,Std,R$ 1.077,2294.0,19/02/2020,4,170861.38,"R$1.028,76",Digital,Sim,0.06,DEB AUTO
1,IND-221302904381,4,padrao,"R$1.164,52",,2021-02-16,1,R$ 102.737,"R$ 479,81",Telefone,1,0.14,DEB AUTO
2,IND-221301446868,1,PREMIUM,"R$ 1.358,76",4925.0,06/12/2012,3,112683.48,"R$667,98",Digital,Em dia,#N/D,Debito_Auto
3,IND-221303000630,1,PADRAO,728.34,1258.0,2022/12/21,2,"R$ 156.010,05","R$800,06",Digital,S,0.0,Debito_Auto
4,IND-221300162937,4,Basica,1628.21,1430.0,2022/7/2,1,101634.15,837.69,Telefone,1,0.171,Debito Automatico


### `num_apolices_ativas` e `tipo_cobertura`

Alguns registros de `tipo_cobertura` vieram com o caractere de acentuação corrompido no arquivo
de origem (ex.: `padr�o`, `b�sica`) — tratados como variantes válidas de `padrao`/`basica`.

In [9]:
# num_apolices_ativas: numérico, faixa observada 1-6, margem de segurança até 10
df_cont["num_apolices_ativas"] = df_cont["num_apolices_ativas"].apply(lambda x: parse_numero(x, 1, 10))

# tipo_cobertura: categórica -> one-hot (inclui variantes com encoding corrompido)
TIPO_COBERTURA_MAP = {
    "padrao": "padrao", "padrão": "padrao", "padr�o": "padrao", "std": "padrao",
    "basica": "basica", "básica": "basica", "b�sica": "basica", "basic": "basica",
    "premium": "premium", "plus": "premium", "prem.": "premium",
}
df_cont = one_hot_encode(df_cont, "tipo_cobertura", TIPO_COBERTURA_MAP)


In [10]:
# valor_premio_anual: monetária, faixa coerente 100-20.000 (sentinelas observados: -10000, 9999999)
df_cont["valor_premio_anual"] = df_cont["valor_premio_anual"].apply(lambda x: parse_money(x, 100, 20_000))

# data_primeira_apolice: normaliza para dd/mm/yyyy
df_cont["data_primeira_apolice"] = df_cont["data_primeira_apolice"].apply(parse_date)

# tempo_cliente_dias: numérico, faixa coerente 0-8.000 dias (sentinelas observados: -30, 99999)
df_cont["tempo_cliente_dias"] = df_cont["tempo_cliente_dias"].apply(lambda x: parse_numero(x, 0, 8_000))

# num_produtos_contratados: numérico, faixa observada 1-7, margem de segurança até 15
df_cont["num_produtos_contratados"] = df_cont["num_produtos_contratados"].apply(lambda x: parse_numero(x, 1, 15))

# valor_cobertura_total: monetária, faixa coerente 30.000-1.200.000 (sentinelas: 999999999, 3999999996)
df_cont["valor_cobertura_total"] = df_cont["valor_cobertura_total"].apply(lambda x: parse_money(x, 30_000, 1_200_000))

# franquia_media: monetária, faixa observada 175-4.800, margem de segurança 100-5.000
df_cont["franquia_media"] = df_cont["franquia_media"].apply(lambda x: parse_money(x, 100, 5_000))

# canal_aquisicao: categórica -> one-hot
CANAL_AQUISICAO_MAP = {"agente": "agente", "digital": "digital", "telefone": "telefone", "indicacao": "indicacao", "indicação": "indicacao"}
df_cont = one_hot_encode(df_cont, "canal_aquisicao", CANAL_AQUISICAO_MAP)

# pagamento_em_dia: sim -> 1, nao -> 0
PAGTO_SIM = {"1", "s", "ok", "sim", "em dia"}
PAGTO_NAO = {"0", "n", "atrasado", "nao", "não"}
df_cont["pagamento_em_dia"] = df_cont["pagamento_em_dia"].apply(lambda x: parse_sim_nao(x, PAGTO_SIM, PAGTO_NAO))

# desconto_aplicado_pct: numérico (percentual em fração 0-1), faixa observada 0-0.25, margem de segurança até 1
df_cont["desconto_aplicado_pct"] = df_cont["desconto_aplicado_pct"].apply(lambda x: parse_numero(x, 0, 1))

# metodo_pagamento: categórica -> one-hot
METODO_PAGAMENTO_MAP = {
    "debito": "debito", "deb auto": "debito", "debito_auto": "debito", "debito automatico": "debito",
    "boleto bancario": "boleto", "boleto": "boleto", "bol": "boleto",
    "cartao": "credito", "cc": "credito", "cartao_credito": "credito", "cartão": "credito",
    "pix": "pix",
}
df_cont = one_hot_encode(df_cont, "metodo_pagamento", METODO_PAGAMENTO_MAP)

df_cont.describe(include="all")


,cod_individuo,num_apolices_ativas,valor_premio_anual,tempo_cliente_dias,data_primeira_apolice,num_produtos_contratados,valor_cobertura_total,franquia_media,pagamento_em_dia,desconto_aplicado_pct,tipo_cobertura_basica,tipo_cobertura_padrao,tipo_cobertura_premium,tipo_cobertura_NaN,canal_aquisicao_agente,canal_aquisicao_digital,canal_aquisicao_indicacao,canal_aquisicao_telefone,canal_aquisicao_NaN,metodo_pagamento_boleto,metodo_pagamento_credito,metodo_pagamento_debito,metodo_pagamento_pix,metodo_pagamento_NaN
count,94000,89338.000000,88933.000000,89128.000000,89351,89281.000000,8.894400e+04,89369.000000,89276.000000,89297.000000,94000.000000,94000.000000,94000.000000,94000.000000,94000.000000,94000.000000,94000.000000,94000.000000,94000.000000,94000.000000,94000.000000,94000.000000,94000.000000,94000.000000
unique,94000,NaN,NaN,NaN,6759,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,IND-221302198005,NaN,NaN,NaN,02/05/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,4501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2.414874,1307.641941,2367.749159,NaN,2.348305,1.483159e+05,1005.015448,0.937486,0.096701,0.315309,0.362596,0.272415,0.049681,0.341074,0.286628,0.133394,0.187915,0.050989,0.236766,0.208915,0.360362,0.144266,0.049691
std,NaN,1.266653,496.242651,1525.099079,NaN,1.535941,7.275891e+04,394.375099,0.242088,0.054412,0.464641,0.480752,0.445205,0.217286,0.474073,0.452188,0.340001,0.390646,0.219977,0.425100,0.406536,0.480108,0.351361,0.217308
min,NaN,1.000000,379.510000,30.000000,NaN,1.000000,4.296330e+04,175.210000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,1.000000,1067.630000,1205.000000,NaN,1.000000,1.158854e+05,707.580000,1.000000,0.051000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,NaN,2.000000,1225.670000,2234.000000,NaN,2.000000,1.362658e+05,959.100000,1.000000,0.090000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,NaN,4.000000,1418.980000,3386.000000,NaN,4.000000,1.602007e+05,1241.430000,1.000000,0.141000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000


In [11]:
print("Linhas antes:", 94000, "| Linhas depois:", len(df_cont))
assert len(df_cont) == 94000, "Nenhuma linha deveria ter sido removida"

df_cont.to_csv("../bases/bases_kaggle/tratadas/contratos_apolices_kaggle_tratada.csv", index=False)
print("Salvo em ../bases/bases_kaggle/tratadas/contratos_apolices_kaggle_tratada.csv")


Linhas antes: 94000 | Linhas depois: 94000


Salvo em ../bases/bases_kaggle/tratadas/contratos_apolices_kaggle_tratada.csv
